# Instructions

In this notebook you have some problems to solve with Python coding. For each problem, explain your code with comments and explain how it works to solve the problem being addressed. You also MUST include the answers you get from running your code. These instructions may change as needed, so we suggest you read these anew on each homework (we will try to announce ahead if they change).

Collaboration is both allowed and encouraged! Using available resources from your peers is both wise and efficient. While you are welcome to work collaboratively with classmates, that means you can work together to solve problems together, not that one person does the work and others copy it. You are also welcome to use the internet as a resource to refresh your memory, clarify concepts, and help with short code snippets. In order to model good practice, you **MUST** cite your sources **at the time of assignment submission** (we will not accept citations sent after-the-fact), and you should not be copying large portions of code wholesale from any source: internet, human, artificial intelligence, or otherwise.

Things that are off limits:

 -    Soliciting (hey, what did you get for question X? Can you send me your code?) and/or copying code and/or answers from another person.

 -    Using sources appropriately but failing to cite them.

 -     Getting any AI, including ChatGPT, Gemini, or CoPilot to do your work for you.


As with every assignment in this course, begin with installing and importing the library designed for this course, ``determinism301`` using the following code in its own cell at the very beginning:
```py
!pip install -qU --extra-index-url https://determinism.data301.download determinism301
import determinism301
```

Before submitting, once you're sure your code is correct, it is recommended that you restart the session kernel under the Runtime/Kernel section and rerun your code with determinism301 to get your final answer to submit.

You **may not** set random_state, seed, or other similar parameters manually for any library unless explicitly instructed. For numpy, you **must** use the old-style ``np.random.uniform``, ``np.random.normal``, and so on, rather than ``np.random.default_rng()`` variants.

Make sure you include **(YOUR NAME, SUBQUESTION)** (e.g. "Jane Smith, 2c") at the start of every code cell.

**You may only use libraries that were imported for you at the top.**

**Note: All plots should have x and y axis labels, and a title at minimum, plus a legend if multiple distinct elements are present.**

# Import Statements

You **MUST** run the below code cells every time you re-open this assignment in order for your results to be reliable

In [ ]:
# InstallBlock
!pip install -qU --extra-index-url https://determinism.data301.download determinism301
import determinism301

In [ ]:
# ImportBlock
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors
import scipy.stats as stats
import math
import time
import copy
from IPython.display import display

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report,
    precision_recall_curve, accuracy_score, f1_score
)
from sklearn.svm import SVC
from sklearn.datasets import make_moons, make_circles

# Problem 1: Gradient Descent for Regression (35 Points)

In this problem, you will implement gradient descent from scratch to fit a linear regression model to a real-world dataset. You will explore how different learning rates and batch sizes affect convergence, and visualize the optimization trajectory on the loss landscape.

We will work with a synthetic housing-style dataset throughout this problem:

```py
# Dataset Generation - DO NOT MODIFY
def generate_regression_data(n_samples=1000, n_features=5, noise_std=2.0):
    X = np.random.normal(0, 1, size=(n_samples, n_features))
    true_weights = np.array([3.5, -2.0, 4.0, 0.0, -1.5])
    true_bias = 7.0
    y = X @ true_weights + true_bias + np.random.normal(0, noise_std, size=n_samples)
    return X, y, true_weights, true_bias

X_raw, y_raw, true_w, true_b = generate_regression_data()

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_raw, test_size=0.2)
```

## 1a: Loss Function and Numerical Gradient (10 Points)

Build a `LinearRegressionGD` class that computes the Mean Squared Error loss and its gradient for linear regression. The model predicts:

$$\hat{y} = X \mathbf{w} + b$$

and the loss is:

$$\mathcal{L}(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Your class should match the following blueprint:

```py
class LinearRegressionGD:
    def __init__(self, n_features):
        # Initialize weights to small random values and bias to 0
        self.w = np.random.normal(0, 0.01, size=n_features)
        self.b = 0.0

    def predict(self, X):
        # Return X @ w + b
        pass

    def loss(self, X, y):
        # Compute and return the MSE loss
        pass

    def analytical_grad(self, X, y):
        # Compute the analytical gradient of the MSE w.r.t. w and b
        # dL/dw = (-2/n) * X^T @ (y - y_hat)
        # dL/db = (-2/n) * sum(y - y_hat)
        # return grad_w, grad_b
        pass

    def numerical_grad(self, X, y, h=1e-5):
        # Compute the numerical gradient using central differences
        # Perturb each element of w and b independently
        # return grad_w, grad_b
        pass
```

Once implemented:
1. Generate the dataset and create an instance of `LinearRegressionGD`.
2. Print the initial loss on the training set.
3. Compute both the analytical and numerical gradients at the initial parameters.
4. Print the maximum absolute difference between them to verify correctness.

Print results in this format:
```py
>>> Initial Training Loss: XX.XXXX
>>> Max |analytical - numerical| gradient difference: X.XXXXXXXX
```

The max gradient difference should be very small (< 1e-5). If it is not, your analytical gradient is likely incorrect.

## 1b: Batch Gradient Descent with ADAM (15 Points)

Add a `fit` method to your `LinearRegressionGD` class that trains the model using mini-batch gradient descent with the ADAM optimizer. Your method should follow this blueprint:

```py
def fit(self, X_train, y_train, X_test, y_test,
        n_epochs=500, batch_size=32, learning_rate=0.01,
        beta1=0.9, beta2=0.999, eps=1e-8):
    n = len(y_train)

    # Initialize ADAM moment vectors for w and b
    m_w = np.zeros_like(self.w)
    v_w = np.zeros_like(self.w)
    m_b = 0.0
    v_b = 0.0
    t_step = 0

    train_loss_history = []
    test_loss_history = []

    for epoch in range(n_epochs):
        # Shuffle training data
        indices = np.random.permutation(n)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        # Mini-batch loop
        for start in range(0, n, batch_size):
            # Extract batch
            # Compute analytical gradient on the batch
            # ADAM update for w and b
            t_step += 1
            # m_w = beta1 * m_w + (1 - beta1) * grad_w  (and similarly for v_w, m_b, v_b)
            # Bias correction
            # Parameter update
            pass

        # Record train and test loss at end of each epoch
        train_loss_history.append(self.loss(X_train, y_train))
        test_loss_history.append(self.loss(X_test, y_test))

    self.train_loss_history = train_loss_history
    self.test_loss_history = test_loss_history
```

After training with the default parameters:
1. Plot the train and test loss curves on the same figure (use log scale for the y-axis).
2. Print the final train and test loss.
3. Print the learned weights and bias, and compare to the true values.

```py
>>> Final Train Loss: X.XXXX
>>> Final Test Loss: X.XXXX
>>> Learned weights: [X.XX, X.XX, X.XX, X.XX, X.XX]
>>> True weights:    [3.5, -2.0, 4.0, 0.0, -1.5]
>>> Learned bias: X.XX | True bias: 7.0
```

Feel free to consult the following plotting code:
```py
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(model.train_loss_history, label='Train Loss')
ax.semilogy(model.test_loss_history, label='Test Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss (log scale)')
ax.set_title('Gradient Descent Training Curve')
ax.legend()
plt.tight_layout()
plt.show()
```

For full credit, write 2-3 sentences describing what you observe about the training curve and how close the learned parameters are to the true ones.

*[Your written response here]*

## 1c: Sensitivity Analysis (10 Points)

Investigate how the learning rate and batch size affect convergence.

For learning rates in `[0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]` and batch sizes in `[16, 32, 64, 128, 256, 800]`:
- Train a fresh `LinearRegressionGD` model for `n_epochs=200` with each combination.
- Record the final test loss.

Produce a heatmap (using `plt.pcolormesh`) of the final test loss across the grid of (learning_rate x batch_size). Use a log color scale (`norm=matplotlib.colors.LogNorm()`).

Print the best combination found:
```py
>>> Best: lr=X.XXXX, batch_size=XXX -> Test Loss: X.XXXXXX
```

Feel free to consult the following plotting code:
```py
learning_rates = [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]
batch_sizes = [16, 32, 64, 128, 256, 800]

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.pcolormesh(loss_grid, norm=matplotlib.colors.LogNorm(), cmap='viridis')
fig.colorbar(im, label='Final Test Loss (log scale)')

for i in range(len(learning_rates)):
    for j in range(len(batch_sizes)):
        ax.text(j + 0.5, i + 0.5, f"{loss_grid[i, j]:.2e}",
                ha='center', va='center', color='red', fontsize=9, fontweight='bold')

ax.set_xticks(np.arange(len(batch_sizes)) + 0.5)
ax.set_xticklabels(batch_sizes)
ax.set_yticks(np.arange(len(learning_rates)) + 0.5)
ax.set_yticklabels(learning_rates)
ax.set_xlabel('Batch Size')
ax.set_ylabel('Learning Rate')
ax.set_title('Sensitivity Analysis: Final Test Loss')
plt.tight_layout()
plt.show()

best_idx = np.unravel_index(np.argmin(loss_grid), loss_grid.shape)
print(f"Best: lr={learning_rates[best_idx[0]]}, batch_size={batch_sizes[best_idx[1]]} -> Test Loss: {loss_grid[best_idx]:.6f}")
```

For full credit, write 2-3 sentences interpreting the heatmap. What trends do you see with respect to learning rate and batch size? Are there any combinations that fail to converge?

*[Your written response here]*

# Problem 2: Regularized Regression (30 Points)

In this problem, you will extend your gradient descent implementation to include L1 (Lasso), L2 (Ridge), and ElasticNet regularization. You will explore how regularization affects model coefficients and use cross-validation to select the best hyperparameters.

We will use a dataset with many features, some of which are irrelevant, to see how regularization helps:

```py
# Dataset Generation - DO NOT MODIFY
def generate_sparse_data(n_samples=500, n_features=50, n_informative=5, noise_std=1.0):
    X = np.random.normal(0, 1, size=(n_samples, n_features))
    true_weights = np.zeros(n_features)
    informative_idx = np.array([0, 5, 12, 23, 41])
    true_weights[informative_idx] = np.array([4.0, -3.0, 2.5, -1.5, 3.0])
    true_bias = 5.0
    y = X @ true_weights + true_bias + np.random.normal(0, noise_std, size=n_samples)
    return X, y, true_weights, true_bias, informative_idx

X_sparse, y_sparse, true_w_sparse, true_b_sparse, info_idx = generate_sparse_data()

# Standardize features
scaler2 = StandardScaler()
X_sparse_scaled = scaler2.fit_transform(X_sparse)

# Train/test split (80/20)
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_sparse_scaled, y_sparse, test_size=0.2)
```

## 2a: Regularized Gradient Descent (15 Points)

Create a `RegularizedRegressionGD` class that extends your gradient descent approach to include regularization. The regularized loss is:

$$\mathcal{L}(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 + \alpha \left[ \rho \|\mathbf{w}\|_1 + \frac{1 - \rho}{2} \|\mathbf{w}\|_2^2 \right]$$

where $\alpha$ controls regularization strength and $\rho$ controls the L1/L2 mix ($\rho = 1$ is Lasso, $\rho = 0$ is Ridge, $0 < \rho < 1$ is ElasticNet).

Your class should follow this blueprint:

```py
class RegularizedRegressionGD:
    def __init__(self, n_features, alpha=0.1, rho=0.5):
        # Initialize weights to small random values and bias to 0
        self.w = np.random.normal(0, 0.01, size=n_features)
        self.b = 0.0
        self.alpha = alpha
        self.rho = rho  # 1.0 = Lasso, 0.0 = Ridge

    def predict(self, X):
        pass

    def loss(self, X, y):
        # MSE + regularization penalty
        # Note: bias is NOT regularized
        pass

    def grad(self, X, y):
        # Analytical gradient of the regularized loss
        # L2 gradient contribution: alpha * (1 - rho) * w
        # L1 gradient contribution: alpha * rho * sign(w)
        # return grad_w, grad_b
        pass

    def fit(self, X_train, y_train, X_test, y_test,
            n_epochs=500, batch_size=32, learning_rate=0.01,
            beta1=0.9, beta2=0.999, eps=1e-8):
        # Same ADAM optimizer loop as Problem 1b,
        # but using the regularized gradient
        # Store self.train_loss_history and self.test_loss_history
        pass
```

Once implemented, train three models on the sparse dataset:
1. **Ridge** ($\rho = 0.0$, $\alpha = 0.1$)
2. **Lasso** ($\rho = 1.0$, $\alpha = 0.1$)
3. **ElasticNet** ($\rho = 0.5$, $\alpha = 0.1$)

For each model:
- Plot the learned weight vector as a bar chart (all 50 features on the x-axis, weight magnitude on the y-axis). Highlight the 5 informative features in a different color.
- Print the test loss and the number of "near-zero" weights (|w| < 0.05).

```py
>>> [Ridge]      Test Loss: X.XXXX | Near-zero weights: XX/50
>>> [Lasso]      Test Loss: X.XXXX | Near-zero weights: XX/50
>>> [ElasticNet] Test Loss: X.XXXX | Near-zero weights: XX/50
```

Feel free to consult the following plotting code:
```py
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
labels = ['Ridge (rho=0)', 'Lasso (rho=1)', 'ElasticNet (rho=0.5)']
models = [model_ridge, model_lasso, model_elastic]

for ax, model, label in zip(axes, models, labels):
    colors = ['tab:orange' if i in info_idx else 'tab:blue' for i in range(len(model.w))]
    ax.bar(range(len(model.w)), model.w, color=colors)
    ax.set_xlabel('Feature Index')
    ax.set_title(label)
    ax.axhline(0, color='black', linewidth=0.5)

axes[0].set_ylabel('Weight Value')
fig.suptitle('Learned Weights Under Different Regularization', fontsize=13)
plt.tight_layout()
plt.show()
```

For full credit, write 3-4 sentences comparing how Ridge, Lasso, and ElasticNet handle the irrelevant features. Which method(s) drive the irrelevant weights closest to zero?

*[Your written response here]*

## 2b: Regularization Path (7.5 Points)

A regularization path shows how the model coefficients change as the regularization strength $\alpha$ varies. This helps us understand the tradeoff between model complexity and regularization.

For **Lasso** ($\rho = 1.0$), sweep $\alpha$ over `np.logspace(-4, 1, 30)` (30 values from 0.0001 to 10). For each $\alpha$, train a fresh `RegularizedRegressionGD` model with `n_epochs=300` and `learning_rate=0.01`. Record the learned weight vector.

Produce a plot where:
- The x-axis is $\alpha$ (log scale).
- The y-axis is the weight value.
- Each of the 50 features is a separate line.
- The 5 informative features are plotted with thicker, distinctly colored lines and labeled in the legend.
- The remaining 45 features are plotted with thin gray lines (no individual legend entries).

Feel free to consult the following plotting code:
```py
alphas = np.logspace(-4, 1, 30)
# weight_paths should be shape (30, 50)

fig, ax = plt.subplots(figsize=(12, 6))
for j in range(weight_paths.shape[1]):
    if j in info_idx:
        ax.plot(alphas, weight_paths[:, j], linewidth=2.5,
                label=f'Feature {j} (true w={true_w_sparse[j]:.1f})')
    else:
        ax.plot(alphas, weight_paths[:, j], color='gray', alpha=0.3, linewidth=0.8)

ax.set_xscale('log')
ax.set_xlabel('Alpha (log scale)')
ax.set_ylabel('Weight Value')
ax.set_title('Lasso Regularization Path')
ax.axhline(0, color='black', linewidth=0.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
```

For full credit, write 2-3 sentences describing what happens to the informative vs. non-informative weights as $\alpha$ increases.

*[Your written response here]*

## 2c: Cross-Validated Hyperparameter Selection (7.5 Points)

Use 5-fold cross-validation to select the best $\alpha$ and $\rho$ for the ElasticNet model on the sparse dataset.

For $\alpha$ in `[0.001, 0.01, 0.05, 0.1, 0.5, 1.0]` and $\rho$ in `[0.0, 0.25, 0.5, 0.75, 1.0]`:
- For each of 5 folds, train a `RegularizedRegressionGD` model with `n_epochs=300`, `learning_rate=0.01` and score the held-out fold.
- Record the mean cross-validation MSE across the 5 folds.

You should use `sklearn.model_selection.KFold` with `n_splits=5` and `shuffle=True`.

Produce a heatmap of the mean CV loss across the ($\alpha$ x $\rho$) grid. Print the best combination:
```py
>>> Best CV: alpha=X.XXX, rho=X.XX -> Mean CV Loss: X.XXXXXX
```

Then, train a final model with the best hyperparameters on the full training set and report:
```py
>>> Final model Test Loss: X.XXXX
>>> Number of near-zero weights (|w| < 0.05): XX/50
```

Feel free to consult the following plotting code:
```py
alphas_cv = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
rhos_cv = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.pcolormesh(cv_loss_grid, norm=matplotlib.colors.LogNorm(), cmap='viridis')
fig.colorbar(im, label='Mean CV Loss (log scale)')

for i in range(len(alphas_cv)):
    for j in range(len(rhos_cv)):
        ax.text(j + 0.5, i + 0.5, f"{cv_loss_grid[i, j]:.3f}",
                ha='center', va='center', color='red', fontsize=10, fontweight='bold')

ax.set_xticks(np.arange(len(rhos_cv)) + 0.5)
ax.set_xticklabels(rhos_cv)
ax.set_yticks(np.arange(len(alphas_cv)) + 0.5)
ax.set_yticklabels(alphas_cv)
ax.set_xlabel('Rho (L1/L2 Mix)')
ax.set_ylabel('Alpha (Regularization Strength)')
ax.set_title('Cross-Validation: Mean Test Loss')
plt.tight_layout()
plt.show()
```

For full credit, write 2-3 sentences on what the cross-validation results tell you about the best regularization strategy for this dataset.

*[Your written response here]*

# Problem 3: Support Vector Machines & Classification Diagnostics (35 Points)

In this problem, you will use Support Vector Machines (SVMs) with different kernels to classify data, visualize decision boundaries, and apply classification diagnostics including confusion matrices, ROC curves, and precision-recall curves.

We will generate two synthetic 2D datasets to build intuition, then apply SVMs to a real-world classification problem.

## 3a: SVM Decision Boundaries (10 Points)

Generate two synthetic datasets using sklearn's `make_moons` and `make_circles`:

```py
# Dataset Generation - DO NOT MODIFY
X_moons, y_moons = make_moons(n_samples=500, noise=0.2)
X_circles, y_circles = make_circles(n_samples=500, noise=0.1, factor=0.4)
```

Write a function that trains an SVM and plots its decision boundary:

```py
def plot_svm_boundary(X, y, kernel, C=1.0, gamma='scale', ax=None, title=''):
    # Fit SVC with the given kernel and parameters
    # Create a mesh grid covering the data range (with some padding)
    # Use model.decision_function() on the mesh to get the decision boundary
    # Plot filled contours for the decision regions
    # Scatter plot the data points, colored by class
    # Highlight the support vectors with a distinct marker (e.g., larger size, black edge)
    # Print the number of support vectors
    pass
```

Create a 2x3 figure (2 datasets x 3 kernels). For each dataset (moons, circles), fit SVMs with kernels `'linear'`, `'rbf'`, and `'poly'` (degree=3) using `C=1.0`. Plot the decision boundary for each.

Print the training accuracy and number of support vectors for each combination:
```py
>>> [Moons, linear]  Accuracy: X.XXXX | Support Vectors: XXX
>>> [Moons, rbf]     Accuracy: X.XXXX | Support Vectors: XXX
>>> [Moons, poly]    Accuracy: X.XXXX | Support Vectors: XXX
>>> [Circles, linear] Accuracy: X.XXXX | Support Vectors: XXX
>>> [Circles, rbf]    Accuracy: X.XXXX | Support Vectors: XXX
>>> [Circles, poly]   Accuracy: X.XXXX | Support Vectors: XXX
```

Feel free to consult the following plotting code:
```py
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
datasets = [('Moons', X_moons, y_moons), ('Circles', X_circles, y_circles)]
kernels = ['linear', 'rbf', 'poly']

for row, (name, X, y) in enumerate(datasets):
    for col, kernel in enumerate(kernels):
        ax = axes[row, col]
        plot_svm_boundary(X, y, kernel=kernel, ax=ax,
                         title=f'{name} - {kernel.upper()} kernel')

plt.tight_layout()
plt.show()
```

For full credit, write 3-4 sentences comparing the decision boundaries. Which kernel(s) work best for each dataset? Why does the linear kernel struggle on these datasets?

*[Your written response here]*

## 3b: SVM on Real Data with Hyperparameter Tuning (10 Points)

Now apply SVMs to a real-world classification task using the breast cancer dataset:

```py
# Dataset Loading - DO NOT MODIFY
bc_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/breast-cancer.csv'
bc_columns = ['Sample_ID', 'Clump_Thickness', 'Uniformity_Cell_Size', 'Uniformity_Cell_Shape',
              'Marginal_Adhesion', 'Single_Epithelial_Cell_Size', 'Bare_Nuclei',
              'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses', 'Class']
bc_data = pd.read_csv(bc_url, names=bc_columns, na_values='?').dropna()
X_bc = bc_data.drop(columns=['Sample_ID', 'Class']).values.astype(float)
y_bc = (bc_data['Class'].values == 4).astype(int)  # 1 = malignant, 0 = benign

# Standardize and split
scaler_bc = StandardScaler()
X_bc_scaled = scaler_bc.fit_transform(X_bc)
X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(X_bc_scaled, y_bc, test_size=0.2)

print(f"Dataset shape: {X_bc.shape}")
print(f"Class distribution: {np.bincount(y_bc)}")
```

Perform a grid search over the following hyperparameters using 5-fold cross-validation (use `sklearn.model_selection.KFold` with `n_splits=5, shuffle=True`):
- `C` in `[0.01, 0.1, 1.0, 10.0, 100.0]`
- `kernel` in `['linear', 'rbf', 'poly']`
- For each fold, fit an `SVC` and compute the accuracy on the validation fold.

Record the mean CV accuracy for each (C, kernel) combination. Print the results as a table and identify the best combination:

```py
>>> Grid Search Results:
>>>           linear    rbf      poly
>>> C=0.01    X.XXXX   X.XXXX   X.XXXX
>>> C=0.1     X.XXXX   X.XXXX   X.XXXX
>>> ...
>>> Best: C=XX.X, kernel=XXXX -> CV Accuracy: X.XXXX
```

Train a final model with the best hyperparameters on the full training set and report the test accuracy:
```py
>>> Final Test Accuracy: X.XXXX
```

## 3c: Classification Diagnostics (15 Points)

Using the best SVM model from 3b, produce a comprehensive set of classification diagnostics on the **test set**.

You will produce a single figure with 3 subplots:

**Subplot 1: Confusion Matrix**
- Use `sklearn.metrics.confusion_matrix` to compute the confusion matrix.
- Display it as a heatmap with `plt.imshow`, annotated with the counts in each cell.
- Label axes as "Predicted" and "Actual" with class names ["Benign", "Malignant"].

**Subplot 2: ROC Curve**
- Use `model.decision_function(X_test)` to get the raw scores.
- Compute and plot the ROC curve using `sklearn.metrics.roc_curve`.
- Shade the area under the curve and display the AUC in the legend.
- Plot the diagonal dashed line representing random chance.

**Subplot 3: Precision-Recall Curve**
- Compute and plot the precision-recall curve using `sklearn.metrics.precision_recall_curve`.
- Display the average precision (AP) in the legend.

Below the plots, print the full classification report using `sklearn.metrics.classification_report`.

Feel free to consult the following plotting code:
```py
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Confusion Matrix
cm = confusion_matrix(y_bc_test, y_pred)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Benign', 'Malignant'])
axes[0].set_yticklabels(['Benign', 'Malignant'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=16)

# Subplot 2: ROC Curve
scores = best_model.decision_function(X_bc_test)
fpr, tpr, _ = roc_curve(y_bc_test, scores)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[1].fill_between(fpr, tpr, alpha=0.2, color='darkorange')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Chance')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Subplot 3: Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_bc_test, scores)
ap = np.trapz(precision, recall)  # Approximate AP
axes[2].plot(recall, precision, color='tab:green', lw=2, label=f'PR curve (AP = {abs(ap):.4f})')
axes[2].fill_between(recall, precision, alpha=0.2, color='tab:green')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_bc_test, y_pred, target_names=['Benign', 'Malignant']))
```

For full credit, write 4-5 sentences analyzing the diagnostic plots. Address the following:
- What do the confusion matrix values tell you about the model's strengths and weaknesses?
- What does the AUC tell you about the model's discriminative ability?
- In a medical diagnosis context, which type of error (false positive vs. false negative) is more dangerous, and how does your model perform on that metric?

*[Your written response here]*